In [0]:
from pathlib import Path
import pickle
from pyspark.sql import functions as F
from pyspark import StorageLevel
import json

In [0]:
# ============================================================
# Dataset Construction (Optimized with Reuse)
# ============================================================

def load_processed_features(spark, db, table_names, run_date, sample_window):
    """
    Load pre-processed user and post features from two-tower model.
    These already have encoded indices and normalized values.
    """
    """Load inference dataframes for a specific date."""
    user_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['user_processed_features']}
        WHERE partition_date between date_sub('{run_date}', {sample_window}) and '{run_date}' 
    """)
    
    post_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['post_processed_features']}
        WHERE partition_date between date_sub('{run_date}', {sample_window}) and '{run_date}' 
    """)
        
    # Cache since they'll be used in joins
    user_df.cache()
    post_df.cache()
    
    return user_df, post_df

def load_labels(spark, db, table_names, run_date, sample_window):
    """
    Load interactions and dwell time data.
    """    
    interactions_df = spark.sql(f"""
        SELECT * FROM {db}.{table_names['interaction_retrieved_candidates']}
        WHERE partition_date BETWEEN date_sub('{run_date}', {sample_window}) AND '{run_date}'
    """)

    dwell_df = spark.sql(f"""
        SELECT * FROM {db}.{table_names['dwell_retrieved_candidates']}
        WHERE partition_date BETWEEN date_sub('{run_date}', {sample_window}) AND '{run_date}'
    """)

    # Cache since they'll be used in joins
    interactions_df.cache()
    dwell_df.cache()
    
    return interactions_df, dwell_df


def build_ranking_base_table(
    spark,
    interactions_df,
    dwell_df,
    user_features_processed_df,
    post_features_processed_df,
    user_id_col,
    post_id_col,
    user_cat_cols,
    user_num_cols,
    post_cat_cols,
    post_num_cols,
    user_emb_col,
    post_emb_col,
    partition_col='partition_date',
    window="1 day",
    alpha=1  # negative down-sampling ratio
):
    """
    Build ranking base table using pre-processed features from two-tower model.
    This avoids re-processing the same features.
    """
    # Cache input dataframes
    interactions_df.cache()
    dwell_df.cache()
    
    # Register as temp views
    interactions_df.createOrReplaceTempView("interactions")
    dwell_df.createOrReplaceTempView("dwell")
    user_features_processed_df.createOrReplaceTempView("user_features_processed")
    post_features_processed_df.createOrReplaceTempView("post_features_processed")

    # Build feature selections from processed dataframes
    # Already have _idx and _norm columns from two-tower processing
    user_feature_cols = (
        [user_id_col + "_idx"] +
        user_cat_cols + [c + "_idx" for c in user_cat_cols] +
        user_num_cols + [c + "_norm" for c in user_num_cols] +
        [user_emb_col]
    )
    
    post_feature_cols = (
        [post_id_col + "_idx"] +
        post_cat_cols + [c + "_idx" for c in post_cat_cols] +
        post_num_cols + [c + "_norm" for c in post_num_cols] +
        [post_emb_col]
    )
    
    user_feature_select = ", ".join([f"uf.{c}" for c in user_feature_cols])
    post_feature_select = ", ".join([f"pf.{c}" for c in post_feature_cols])

    # Optimized query using pre-processed features
    query = f"""
        WITH base AS (
            SELECT *,
                   CASE WHEN interaction = 'view' THEN 1 ELSE 0 END AS label_click,
                   CASE WHEN interaction = 'like' THEN 1 ELSE 0 END AS label_like,
                   CASE WHEN interaction = 'comment' THEN 1 ELSE 0 END AS label_comment,
                   CASE WHEN interaction = 'share' THEN 1 ELSE 0 END AS label_share
            FROM interactions
        ),
        
        base_with_dwell AS (
            SELECT b.*,
                   COALESCE(
                       CASE 
                           WHEN d.dwell_date BETWEEN b.{partition_col} AND b.{partition_col} + INTERVAL {window}
                           THEN d.dwell_time/1000 
                       END,
                       0
                   ) AS label_dwell_time
            FROM base b
            LEFT JOIN (
                SELECT user_id, post_id, dwell_time, {partition_col} AS dwell_date
                FROM dwell
            ) d
            ON b.user_id = d.user_id AND b.post_id = d.post_id
        ),
        
        positive_samples AS (
            SELECT * FROM base_with_dwell
            WHERE label_click = 1 OR label_like = 1 OR label_comment = 1 
               OR label_share = 1 OR label_dwell_time > 0
        ),
        
        negative_samples AS (
            SELECT * FROM base_with_dwell
            WHERE label_click = 0 AND label_like = 0 AND label_comment = 0 
               AND label_share = 0 AND (label_dwell_time = 0 OR label_dwell_time IS NULL)
               AND rand() <= {alpha}
        ),
        
        base_downsample AS (
            SELECT * FROM positive_samples
            UNION ALL
            SELECT * FROM negative_samples
        )
        
        SELECT 
            bd.user_id,
            bd.post_id,
            bd.{partition_col},
            bd.label_click,
            bd.label_like,
            bd.label_comment,
            bd.label_share,
            bd.label_dwell_time,
            {user_feature_select},
            {post_feature_select}
        FROM base_downsample bd
        LEFT JOIN user_features_processed uf
            ON bd.user_id = uf.user_id
            AND bd.{partition_col} = uf.{partition_col}
        LEFT JOIN post_features_processed pf
            ON bd.post_id = pf.post_id
            AND bd.{partition_col} = pf.{partition_col}
    """
    
    result = spark.sql(query)
    
    # Unpersist cached dataframes
    interactions_df.unpersist()
    dwell_df.unpersist()
    
    return result


def remove_rows_with_null(df, columns):
    """Remove rows with null values efficiently."""
    if not columns:
        return df
    
    valid_columns = [col for col in columns if col in df.columns]
    
    if not valid_columns:
        return df
    
    return df.dropna(subset=valid_columns)


# ============================================================
# Metadata and Statistics
# ============================================================

def calculate_class_ratios(df, label_cols):
    """
    Calculate class imbalance ratios efficiently using single aggregation.
    """
    valid_columns = [col for col in label_cols if col in df.columns]
    
    if not valid_columns:
        return {}
    
    # Single aggregation pass for all labels
    agg_exprs = []
    for column in valid_columns:
        agg_exprs.extend([
            F.sum(F.col(column)).alias(f"{column}_sum"),
            F.count(F.col(column)).alias(f"{column}_count")
        ])
    
    result = df.agg(*agg_exprs).first()
    
    # Calculate ratios
    ratios = {}
    for column in valid_columns:
        ones_count = result[f"{column}_sum"] or 0
        total_count = result[f"{column}_count"] or 0
        zeros_count = total_count - ones_count
        
        if zeros_count == 0:
            ratio = float('inf') if ones_count > 0 else 0.0
        else:
            ratio = ones_count / zeros_count
            
        ratios[column] = ratio
    
    return ratios


def get_categorical_dimensions_from_processed(df, cat_cols):
    """
    Get categorical dimensions from already-processed dataframe.
    """
    # Single aggregation for all categorical columns
    agg_exprs = [F.max(F.col(col + "_idx")).alias(col) for col in cat_cols]
    result = df.agg(*agg_exprs).first()
    
    # Max index + 1 gives us the dimension (since we shifted by 1)
    cat_dims = {col: result[col] + 1 for col in cat_cols}
    
    return cat_dims


def save_ranking_metadata(df, label_cols, metadata_dir):
    """
    Save only ranking-specific metadata.
    Feature processing metadata already exists from two-tower model.
    """
    Path(metadata_dir).mkdir(parents=True, exist_ok=True)
    
    # Calculate class ratios (ranking-specific)
    class_ratios = calculate_class_ratios(df, label_cols)
    # Remove 'label_' prefix for consistency
    class_ratios = {k.replace('label_', ''): v for k, v in class_ratios.items() 
                   if k != 'label_dwell_time'}
    
    # Save ranking-specific metadata
    pickle.dump(class_ratios, open(Path(metadata_dir) / 'class_ratios.pkl', 'wb'))
    # pickle.dump(cat_dims, open(Path(metadata_dir) / 'categorical_dimensions.pkl', 'wb'))
    
    print(f"Saved ranking-specific metadata to {metadata_dir}")
    print(f"  - Class ratios: {class_ratios}")
    # print(f"  - Categorical dimensions: {cat_dims}")
    
    return None


def determine_target_partitions(row_count, rows_per_partition=250000, min_partitions=64, max_partitions=512):
    if row_count <= 0:
        return min_partitions

    partition_count = (row_count + rows_per_partition - 1) // rows_per_partition
    return max(min_partitions, min(max_partitions, partition_count))


# ============================================================
# Main Processing Function
# ============================================================

def process_ranking_dataframe(df, 
                              user_id_col, post_id_col,
                              user_cat_cols, post_cat_cols,
                              user_num_cols, post_num_cols,
                              user_emb_col, post_emb_col,
                              label_cols,
                              parquet_dir,
                              metadata_dir,
                              row_count):
    """
    Process ranking model data using pre-processed features.
    No need to re-encode or normalize - already done in two-tower processing!
    """
    # Cache for statistics collection
    df.cache()
    
    # All features are already processed, just need to organize columns
    cat_cols = user_cat_cols + post_cat_cols
    num_cols = user_num_cols + post_num_cols
    
    # # Get categorical dimensions from existing indices
    # print("Extracting categorical dimensions...")
    # cat_dims = get_categorical_dimensions_from_processed(df, cat_cols)
    
    # # Collect statistics in single pass
    # print("Collecting statistics...")
    # stats = df.agg(
    #     F.countDistinct(user_id_col + "_idx").alias("n_users"),
    #     F.countDistinct(post_id_col + "_idx").alias("n_posts")
    # ).first()
    
    # n_users = stats['n_users']
    # n_posts = stats['n_posts']
    
    # Save ranking-specific metadata
    print("Saving ranking metadata...")
    save_ranking_metadata(df, label_cols, metadata_dir)
    
    # Select final columns for ranking model
    select_cols = [
        user_id_col, user_id_col + '_idx',
        post_id_col, post_id_col + '_idx',
        *user_cat_cols, *[c + "_idx" for c in user_cat_cols],
        *post_cat_cols, *[c + "_idx" for c in post_cat_cols],
        *user_num_cols, *[c + "_norm" for c in user_num_cols],
        *post_num_cols, *[c + "_norm" for c in post_num_cols],
        user_emb_col, post_emb_col,
        *label_cols
    ]
    
    df_final = df.select(*select_cols)
    
    # Write to parquet with adaptive partitioning and compression
    print("Writing processed data...")
    num_partitions = determine_target_partitions(row_count)
    print(f"Writing with {num_partitions} partitions")
    
    (df_final
     .repartition(num_partitions)
     .write
     .mode("overwrite")
     .option("compression", "snappy")
     .parquet(parquet_dir))
    
    # Unpersist
    df.unpersist()
    
    return None

In [0]:
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

In [0]:
# Configuration
user_id_col = 'user_id'
post_id_col = 'post_id'
partition_col = 'partition_date'

user_cat_cols = ["ip_province", "vehicle_series", "platform"]
user_num_cols = ["months_since_registration", "months_since_consent", "identity", 
                "is_koc", "has_profile_photo", "community_visits", "posts_published", 
                "posts_viewed", "posts_liked", "posts_commented", "posts_shared", 
                "users_followed", "tribes_joined"]

post_cat_cols = ["post_type"]
post_num_cols = ["days_since_published", "is_ugc", "is_sink", "has_pic", "has_video",
                    "views", "likes", "comments", "shares", "dwell_time"]

user_emb_col = 'lastn_embs'
post_emb_col = 'post_content_emb'
label_cols = ['label_click', 'label_like', 'label_comment', 'label_share', 'label_dwell_time']

user_feature_cols = user_cat_cols + user_num_cols + [user_emb_col]
post_feature_cols = post_cat_cols + post_num_cols + [post_emb_col]

ranking_parquet_dir = model_config['RANKING_TRAIN_PARQUET_PATH']
ranking_metadata_dir = model_config['RANKING_METADATA_PATH']

neg_ratio = model_config['neg_sample_ratio']
sample_window = model_config['data_sample_window']

In [0]:
# ========================================
# Step 1: Load pre-processed features from two-tower model
# ========================================
print("="*60)
print("Step 1: Loading pre-processed features from two-tower model")
print("="*60)

# Load processed user and post features
# Note: Adjust these paths based on how two-tower model saves data
# If saved in separate directories, use those paths
user_features_processed, post_features_processed = load_processed_features(
    spark, db, table_names, run_date, sample_window
)
print("Features loaded.")

# ========================================
# Step 2: Load interaction and dwell data
# ========================================
print("\n" + "="*60)
print("Step 2: Loading interaction and dwell data")
print("="*60)

interactions_df, dwell_df = load_labels(spark, db, table_names, run_date, sample_window)
print("Labels loaded.")

# ========================================
# Step 3: Build ranking base table
# ========================================
print("\n" + "="*60)
print("Step 3: Building ranking base table")
print("="*60)

base_table = build_ranking_base_table(
    spark,
    interactions_df,
    dwell_df,
    user_features_processed,
    post_features_processed,
    user_id_col,
    post_id_col,
    user_cat_cols,
    user_num_cols,
    post_cat_cols,
    post_num_cols,
    user_emb_col,
    post_emb_col,
    partition_col=partition_col,
    window="1 day",
    alpha=neg_ratio
)

# Remove nulls (should be minimal since features are pre-processed)
print("Removing null values...")
# Only check _idx and _norm columns since raw values might have been dropped
check_cols = (
    [c + "_idx" for c in user_cat_cols + post_cat_cols] +
    [c + "_norm" for c in user_num_cols + post_num_cols]
)
base_table = remove_rows_with_null(base_table, check_cols)
base_table = base_table.persist(StorageLevel.MEMORY_AND_DISK)
base_row_count = base_table.count()

print(f"Base table size: {base_row_count} rows")

# ========================================
# Step 4: Process and save ranking data
# ========================================
print("\n" + "="*60)
print("Step 4: Processing ranking data")
print("="*60)

process_ranking_dataframe(
    base_table,
    user_id_col, post_id_col,
    user_cat_cols, post_cat_cols,
    user_num_cols, post_num_cols,
    user_emb_col, post_emb_col,
    label_cols,
    ranking_parquet_dir,
    ranking_metadata_dir,
    base_row_count
)

# Calculate and display class ratios
class_ratios = calculate_class_ratios(base_table, label_cols)
class_ratios_display = {k.replace('label_', ''): v for k, v in class_ratios.items()}

# ========================================
# Final Summary
# ========================================
print("\n" + "="*60)
print("RANKING MODEL DATA PROCESSING COMPLETE!")
print("="*60)
print(f"Class ratios:          {class_ratios_display}")
print(f"\nData saved to:         {ranking_parquet_dir}")
print(f"Metadata saved to:     {ranking_metadata_dir}")
print("="*60)

# Unpersist any remaining cached dataframes
user_features_processed.unpersist()
post_features_processed.unpersist()
base_table.unpersist()

In [0]:
# pickle.dump(metadata, open(Path(ranking_metadata_dir) / 'metadata_stats.pkl', 'wb'))